# MediRead — Qwen3-0.6B + LoRA → `.litertlm` (Colab)

**STATUS: NOT EXECUTED.** No run has completed; nothing here proves the export works.

## Colab prerequisites

1. **Google Drive** (recommended): put `merged-fp16.zip` at
   `MyDrive/mediread/merged-fp16.zip`. Colab mounts Drive in §4; a direct upload is the
   fallback.
2. **Internet is available by default** in Colab; it is needed for `uv`, pip and the
   pinned GitHub metadata file.
3. **Runtime: CPU** is enough; conversion is CPU-bound.

## Environment

The Colab kernel may not be Python 3.11, so the notebook builds its own
`/content/.venv311` (exporter) and `/content/.venv311-lm` (runtime) with `uv`. The
kernel only orchestrates subprocesses.

| component | pin |
|---|---|
| Python (both venvs) | 3.11 |
| torch | 2.14.1 |
| transformers | 5.22.2 |
| tensorflow | 2.21.0 |
| litert-torch | 0.10.0 |
| litert-lm-builder | 0.18.1 |
| litert-lm (runtime) | 0.19.1 |
| ai-edge-litert | 2.3.0 |
| protobuf | 6.31.1 |
| sentencepiece | 0.2.1 |

Qwen3 metadata is pinned to commit `02d7e8d8…` with SHA-256 `f132370a…`.

**Experiment: unquantized export.** `--quantization_recipe=none` (verified against the
exporter source, not assumed) and `--model_dtype=bfloat16` weights. No INT4/INT8 weight
quantization is used. float32 weights would be ~2.4 GB, above the 1.5 GB limit; bfloat16
is the highest supported floating-point weight precision that fits.

Colab sessions are ephemeral: §11 copies the artifact and manifest to Drive when it is
mounted.


## 0. Run-state and check registry

In [ ]:
import datetime, platform, sys

CHECKS: list[dict] = []
MANIFEST: dict = {}
ARTIFACT = None
EXPORT_CMD: list[str] = []


def record(name: str, ok: bool, detail: str = "") -> bool:
    """Record a check; re-running a cell replaces that check instead of stacking a stale FAIL."""
    entry = {"name": name, "status": "PASS" if ok else "FAIL", "detail": str(detail)}
    for existing in CHECKS:
        if existing["name"] == name:
            existing.update(entry)
            break
    else:
        CHECKS.append(entry)
    print(("PASS " if ok else "FAIL ") + name + (f"  [{detail}]" if detail else ""))
    return ok


def gate(start: int) -> None:
    """Stop the notebook if anything recorded since `start` failed."""
    failed = [c for c in CHECKS[start:] if c["status"] == "FAIL"]
    if failed:
        for item in failed:
            print(f"  gate failure: {item['name']} [{item['detail']}]")
        raise SystemExit("gate failed: " + ", ".join(c["name"] for c in failed))


print("notebook status: NOT EXECUTED at authoring; kernel", platform.python_version(),
      datetime.datetime.now(datetime.timezone.utc).isoformat())

## 1. Kernel, disk and internet checks

In [ ]:
import shutil, socket
start = len(CHECKS)

print("kernel python:", platform.python_version())
print("platform     :", platform.platform())
free_gb = shutil.disk_usage("/content").free / 1e9
print(f"free /content: {free_gb:.1f} GB")

try:
    socket.create_connection(("pypi.org", 443), timeout=5).close()
    internet = True
except OSError as exc:
    internet = False
    print("internet check failed:", exc)

record("kernel python >= 3.10", sys.version_info >= (3, 10), platform.python_version())
record("free disk > 12 GB", free_gb > 12, f"{free_gb:.1f} GB")
record("internet enabled", internet, "enable it in Colab notebook settings")
MANIFEST["kernel_python"] = platform.python_version()
MANIFEST["platform"] = platform.platform()
gate(start)

## 2. Build the Python 3.11 export venv

`uv` downloads a standalone CPython 3.11 into `/content`, independent of the
Colab kernel version. Resolver output is printed; a failure stops the notebook here.

In [ ]:
import subprocess, sys
from pathlib import Path

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "uv"], check=True)

# Export venv: converter only. NO litert-lm here, because litert-lm==0.19.1
# hard-pins litert-lm-builder==0.19.1, which lacks TfLiteModelType.
VENV = Path("/content/.venv311")
subprocess.run(["uv", "venv", "--clear", "--python", "3.11", str(VENV)], check=True)
VENV_PY = VENV / "bin" / "python"

# Runtime venv: the exact CLI/runtime version the Android app ships (0.19.1).
LM_VENV = Path("/content/.venv311-lm")
subprocess.run(["uv", "venv", "--clear", "--python", "3.11", str(LM_VENV)], check=True)
LM_PY = LM_VENV / "bin" / "python"

print(subprocess.run([str(VENV_PY), "--version"], capture_output=True, text=True).stdout.strip())

EXPORT_PINS = [
    "torch==2.14.1",
    "transformers==5.22.2",
    "litert-torch==0.10.0",
    "litert-lm-builder==0.18.1",
    "tensorflow==2.21.0",
    "ai-edge-litert==2.3.0",
    "sentencepiece==0.2.1",
    "safetensors",
    "numpy<2",
    "protobuf==6.31.1",
]
install = subprocess.run(["uv", "pip", "install", "--python", str(VENV_PY), *EXPORT_PINS],
                         capture_output=True, text=True)
print(install.stdout[-4000:])
print(install.stderr[-4000:])
record("export venv created (Python 3.11)", VENV_PY.exists())
record("export dependencies installed (builder 0.18.1)", install.returncode == 0, f"rc={install.returncode}")

runtime = subprocess.run(["uv", "pip", "install", "--python", str(LM_PY), "litert-lm==0.19.1"],
                         capture_output=True, text=True)
print(runtime.stdout[-3000:])
print(runtime.stderr[-3000:])
record("runtime venv has litert-lm 0.19.1", runtime.returncode == 0, f"rc={runtime.returncode}")
if install.returncode != 0 or runtime.returncode != 0:
    raise SystemExit("dependency install failed")

## 3. Hard version gate (inside the export venv)

In [ ]:
import json, subprocess
start = len(CHECKS)

EXPORT_EXPECTED = {
    "torch": "2.14.1",
    "transformers": "5.22.2",
    "tensorflow": "2.21.0",
    "litert-torch": "0.10.0",
    "litert-lm-builder": "0.18.1",
    "ai-edge-litert": "2.3.0",
    "sentencepiece": "0.2.1",
}
RUNTIME_EXPECTED = {"litert-lm": "0.19.1"}


def versions(python: Path, packages: list[str]) -> dict:
    probe = (
        "import importlib.metadata as md, json, platform\n"
        "def v(n):\n"
        "    try:\n"
        "        return md.version(n)\n"
        "    except Exception:\n"
        "        return '<missing>'\n"
        "print(json.dumps({'python': platform.python_version(), 'numpy': v('numpy'), "
        + ", ".join(f"'{pkg}': v('{pkg}')" for pkg in packages)
        + "}))"
    )
    done = subprocess.run([str(python), "-c", probe], capture_output=True, text=True)
    print(done.stdout, done.stderr[-1000:])
    record(f"version probe ran ({python.parent.parent.name})", done.returncode == 0, f"rc={done.returncode}")
    return json.loads(done.stdout) if done.returncode == 0 else {}


export_versions = versions(VENV_PY, list(EXPORT_EXPECTED) + ["litert-lm", "protobuf"])
absent = export_versions.pop("litert-lm", "<missing>")
record("export venv has no litert-lm (isolation)", absent == "<missing>", str(absent))
record("export venv python is 3.11", str(export_versions.get("python", "")).startswith("3.11"),
       export_versions.get("python"))
for package, expected in EXPORT_EXPECTED.items():
    record(f"pin {package}=={expected}", export_versions.get(package) == expected, export_versions.get(package))
record("numpy major < 2", str(export_versions.get("numpy", "")).split(".")[0] == "1", export_versions.get("numpy"))
protobuf_version = str(export_versions.get("protobuf", "0"))
record("protobuf >= 6.31.1 (builder gencode 6.31.1)",
       tuple(int(part) for part in protobuf_version.split(".")[:3]) >= (6, 31, 1), protobuf_version)

runtime_versions = versions(LM_PY, list(RUNTIME_EXPECTED))
for package, expected in RUNTIME_EXPECTED.items():
    record(f"runtime pin {package}=={expected}", runtime_versions.get(package) == expected,
           runtime_versions.get(package))

MANIFEST["export_python"] = export_versions.get("python")
MANIFEST["packages"] = {"export": export_versions, "runtime": runtime_versions}
gate(start)

## 4. Mount Google Drive (recommended transfer)

Colab cannot read your Mac directly. Upload `merged-fp16.zip` to
`MyDrive/mediread/` (recommended), or use the upload button in §5 once Drive mount is
skipped.


In [ ]:
start = len(CHECKS)
USE_DRIVE = True
try:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_ok = True
except Exception as exc:  # noqa: BLE001
    drive_ok = False
    print("Drive mount failed:", exc)
    USE_DRIVE = False
record("Google Drive mounted", drive_ok, "recommended checkpoint transfer")
gate(start)


## 5. Locate the merged checkpoint

Drive is preferred: `/content/drive/MyDrive/mediread/merged-fp16.zip`. A direct
upload of the same zip is the fallback. Both a zip and an extracted directory are
accepted; §6 verifies the SHA-256 of the checkpoint files either way.


In [ ]:
start = len(CHECKS)
from pathlib import Path

DRIVE_ZIP = Path("/content/drive/MyDrive/mediread/merged-fp16.zip")
CANDIDATES = [
    DRIVE_ZIP,
    Path("/content/merged-fp16.zip"),
    Path("/content/merged-fp16"),
]

SOURCE = None
for candidate in CANDIDATES:
    if candidate.is_file() and candidate.suffix == ".zip":
        SOURCE = candidate
        break
    if candidate.is_dir() and (candidate / "model.safetensors").exists():
        SOURCE = candidate
        break

if SOURCE is None:
    from google.colab import files
    uploaded = files.upload()  # choose merged-fp16.zip
    candidate = Path(next(iter(uploaded)))
    if candidate.suffix == ".zip":
        SOURCE = candidate
    elif (candidate.parent / "model.safetensors").exists():
        SOURCE = candidate.parent

record("checkpoint source found", SOURCE is not None, str(SOURCE))
assert SOURCE is not None, (
    "Put merged-fp16.zip at MyDrive/mediread/merged-fp16.zip, or upload it here."
)
print("using:", SOURCE, "(dir)" if SOURCE.is_dir() else "(zip)")
gate(start)


## 5. Extract and verify the fine-tuned checkpoint

In [ ]:
import hashlib, json, shutil, zipfile
start = len(CHECKS)

EXPECTED_FILES = {
    "model.safetensors": ("a5f304b879337c79dba317db20502f5a07cb8cb72bea5291ae8631ef90fdb364", 1192134784),
    "config.json": ("4055e1b5a35e961b94ecaa29e2d7ae6ebccb51fa92c510a6b8c010a814cf7ac8", 1416),
}
free_gb = shutil.disk_usage("/content").free / 1e9
record("free disk > 10 GB before extraction", free_gb > 10, f"{free_gb:.1f} GB")
assert free_gb > 10, "not enough disk for extraction"

CKPT = Path("/content/checkpoint")
if CKPT.exists():
    shutil.rmtree(CKPT)
CKPT.mkdir(parents=True)
if SOURCE.is_dir():
    shutil.copytree(SOURCE, CKPT)
else:
    with zipfile.ZipFile(SOURCE) as archive:
        archive.extractall(CKPT)
nested = [p for p in CKPT.iterdir() if p.is_dir()]
if nested and not (CKPT / "model.safetensors").exists():
    for item in nested[0].iterdir():
        shutil.move(str(item), CKPT / item.name)


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


checkpoint_hashes = {}
for name, (expected_sha, expected_bytes) in EXPECTED_FILES.items():
    path = CKPT / name
    exists = path.exists()
    record(f"checkpoint file {name} exists", exists)
    if not exists:
        continue
    actual_sha = sha256(path)
    checkpoint_hashes[name] = {"sha256": actual_sha, "bytes": path.stat().st_size}
    record(f"{name} SHA-256 matches fine-tuned model", actual_sha == expected_sha, actual_sha[:16] + "…")
    record(f"{name} size matches", path.stat().st_size == expected_bytes, f"{path.stat().st_size} bytes")

config = json.loads((CKPT / "config.json").read_text())
record("architecture is Qwen3ForCausalLM", config["architectures"] == ["Qwen3ForCausalLM"], config["architectures"])
record("28 layers / hidden 1024", config["num_hidden_layers"] == 28 and config["hidden_size"] == 1024)
record("tokenizer files present", (CKPT / "tokenizer.json").exists() and (CKPT / "chat_template.jinja").exists())
MANIFEST["checkpoint"] = checkpoint_hashes
gate(start)

## 6a. Tokenizer config compatibility (working copy)

The merged checkpoint was saved with transformers 5.x, which writes
`extra_special_tokens` as a **list**. Pinned `transformers==5.22.2` expects a **dict**
and raises `AttributeError: 'list' object has no attribute 'keys'`. The exporter loads
the tokenizer from `--model`, so it would fail too.

This cell builds `checkpoint-fixed/`: symlinks to the model files plus a sanitized
`tokenizer_config.json`. **The original checkpoint is not modified.**


In [ ]:
start = len(CHECKS)
import json, shutil
from pathlib import Path

CKPT_FIXED = Path("/content") / "checkpoint-fixed"
if CKPT_FIXED.exists():
    shutil.rmtree(CKPT_FIXED)
CKPT_FIXED.mkdir(parents=True)

for name in ("model.safetensors", "config.json", "generation_config.json",
             "tokenizer.json", "chat_template.jinja", "special_tokens_map.json"):
    source = CKPT / name
    if source.exists():
        (CKPT_FIXED / name).symlink_to(source)

config = json.loads((CKPT / "tokenizer_config.json").read_text())
extra = config.get("extra_special_tokens")
sanitized = isinstance(extra, list)
if sanitized:
    config["extra_special_tokens"] = {}
(CKPT_FIXED / "tokenizer_config.json").write_text(json.dumps(config, indent=2))
record("tokenizer_config sanitized (list -> dict)", True,
       f"extra_special_tokens was {type(extra).__name__}")

probe = (
    "import sys; from transformers import AutoTokenizer;"
    "tok = AutoTokenizer.from_pretrained(sys.argv[1]);"
    "print(tok.__class__.__name__, len(tok), tok.eos_token_id, tok.is_fast)"
)
check = subprocess.run([str(VENV_PY), "-c", probe, str(CKPT_FIXED)], capture_output=True, text=True)
print(check.stdout, check.stderr[-1500:])
record("fixed checkpoint tokenizer loads under the pinned transformers", check.returncode == 0, f"rc={check.returncode}")
fields = check.stdout.split()
record("fixed tokenizer is fast Qwen2 with EOS 151645",
       len(fields) >= 4 and fields[0] in ("Qwen2Tokenizer", "Qwen2TokenizerFast")
       and fields[2] == "151645" and fields[3] == "True", check.stdout.strip())
MANIFEST["tokenizer_compat"] = {"sanitized": sanitized,
                                "extra_special_tokens_was": type(extra).__name__,
                                "probe": check.stdout.strip()}
gate(start)


## 6b. Checkpoint audit and HF reference (before export)

Tokenizes the merged checkpoint, checks EOS/vocab/round-trips, and runs the three
acceptance prompts through the HF model greedily. The exported bundle is compared
against these texts later, so a broken checkpoint stops the run here.


In [ ]:
start = len(CHECKS)
from pathlib import Path
import json, subprocess

AUDIT_SCRIPT = Path("/content/checkpoint_checks.py")
AUDIT_SCRIPT.write_text(r'''
import json, sys
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

CKPT = Path(sys.argv[1])
OUT = Path(sys.argv[2])
PROMPTS = [
    "Explain what a blood glucose test measures in one sentence.",
    "Explain what a haemoglobin test measures in one sentence.",
    "Explain what a cholesterol test measures in one sentence.",
]

tok = AutoTokenizer.from_pretrained(CKPT)
if not getattr(tok, "is_fast", False):
    raise SystemExit("expected a fast tokenizer (tokenizer.json)")

round_trips = {}
for text in ("Haemoglobin 13.5-17.5 g/dL", "HbA1c 5.7 %", "caf\u00e9 na\u00efve"):
    ids = tok.encode(text, add_special_tokens=False)
    round_trips[text] = {"ids": ids, "exact": tok.decode(ids) == text}

audit = {
    "tokenizer_class": type(tok).__name__,
    "vocab_size": len(tok),
    "eos_id": tok.eos_token_id,
    "eos_token": tok.eos_token,
    "pad_id": tok.pad_token_id,
    "bos_id": tok.bos_token_id,
    "template_markers": {
        "im_start": "<|im_start|>" in (tok.chat_template or ""),
        "im_end": "<|im_end|>" in (tok.chat_template or ""),
        "enable_thinking": "enable_thinking" in (tok.chat_template or ""),
    },
    "round_trips": round_trips,
}

model = AutoModelForCausalLM.from_pretrained(CKPT, torch_dtype=torch.float32).eval()
records = []
for prompt in PROMPTS:
    rendered = tok.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False,
    )
    inputs = tok(rendered, return_tensors="pt")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, do_sample=False,
                             pad_token_id=tok.pad_token_id)
    ids = out[0][inputs["input_ids"].shape[1]:].tolist()
    records.append({
        "prompt": prompt,
        "rendered_prompt": rendered,
        "prompt_tokens": inputs["input_ids"][0].tolist(),
        "generated_ids": ids,
        "stopped_on_eos": bool(ids) and ids[-1] == tok.eos_token_id,
        "text": tok.decode(ids, skip_special_tokens=True).strip(),
    })

OUT.write_text(json.dumps(audit, indent=2))
OUT.with_name("hf_reference.json").write_text(json.dumps({"records": records}, indent=2))
print(json.dumps({
    "tokenizer": audit["tokenizer_class"],
    "eos": audit["eos_id"],
    "round_trips_ok": all(v["exact"] for v in round_trips.values()),
    "records": [{"ids": len(r["generated_ids"]), "eos": r["stopped_on_eos"],
                 "text": r["text"]} for r in records],
}, indent=2))
''')

checks = subprocess.run(
    [str(VENV_PY), str(AUDIT_SCRIPT), str(CKPT_FIXED), "/content/audit.json"],
    capture_output=True, text=True,
)
print(checks.stdout[-3000:])
print(checks.stderr[-3000:])
record("checkpoint audit + HF reference ran", checks.returncode == 0, f"rc={checks.returncode}")

audit = json.loads(Path("/content/audit.json").read_text()) if Path("/content/audit.json").exists() else {}
reference_path = Path("/content/hf_reference.json")
reference = json.loads(reference_path.read_text())["records"] if reference_path.exists() else []

record("fast tokenizer loaded",
       audit.get("tokenizer_class") in ("Qwen2Tokenizer", "Qwen2TokenizerFast"),
       audit.get("tokenizer_class"))
record("EOS is <|im_end|> (151645)", audit.get("eos_id") == 151645, audit.get("eos_id"))
record("tokenizer round trips are exact",
       bool(audit.get("round_trips")) and all(v["exact"] for v in audit["round_trips"].values()))
record("chat template has im_start/im_end/enable_thinking",
       all(audit.get("template_markers", {}).values()), audit.get("template_markers"))
record("three HF references produced", len(reference) == 3, str(len(reference)))
for item in reference:
    text = item["text"]
    ascii_ratio = sum(1 for ch in text if ord(ch) < 128) / max(len(text), 1)
    record(f"HF reference coherent: {item['prompt'][:32]}...",
           item["stopped_on_eos"] and len(text) > 20 and ascii_ratio >= 0.8,
           f"eos={item['stopped_on_eos']} ascii={ascii_ratio:.2f}")
MANIFEST["checkpoint_audit"] = audit
MANIFEST["hf_reference"] = reference
gate(start)


## 6c. Qwen3 metadata (commit-pinned, hash-verified)

In [ ]:
import hashlib, subprocess
start = len(CHECKS)

META_COMMIT = "02d7e8d80cb75cbb03d8cba4c63f51e2eb318ec5"
META_SHA256 = "f132370ab5ec50eeea6d38cc3b172339202e130519323067f6e82a7dc438405f"
META_BYTES = 4152
META_URL = ("https://raw.githubusercontent.com/google-ai-edge/LiteRT-LM/"
            + META_COMMIT + "/models/qwen3/LlmMetadataProto.pbtext")
META = Path("/content/qwen3_metadata.pbtext")
subprocess.run(["curl", "-fsSL", META_URL, "-o", str(META)], check=True)

meta_sha = hashlib.sha256(META.read_bytes()).hexdigest()
record("Qwen3 metadata bytes", META.stat().st_size == META_BYTES, META.stat().st_size)
record("Qwen3 metadata SHA-256 matches pinned commit", meta_sha == META_SHA256, meta_sha[:16] + "…")
MANIFEST["metadata"] = {"repo": "google-ai-edge/LiteRT-LM", "commit": META_COMMIT,
                        "sha256": meta_sha, "bytes": META.stat().st_size}
gate(start)

## 7a. Quantization semantics (source-checked)\n\nThe exporter's CLI default is a quantized recipe, so `--quantization_recipe=none`\nmust be passed explicitly; `maybe_quantize_model` returns the model untouched for\n`none`/`None`/`""`. Weights are exported in **bfloat16** (`--model_dtype=bfloat16`),\nthe highest supported floating-point precision under the 1.5 GB limit (float32 would\nbe ~2.4 GB). No integer weight quantization is used.\n

In [ ]:
start = len(CHECKS)
import json, subprocess

# Do NOT assume that omitting --quantization_recipe disables quantization: the CLI
# default is a quantized recipe. The exporter's own source keeps the model untouched
# only for an explicit "none"/None/"" recipe.
probe = (
    "import json\n"
    "from litert_torch.generative.export_hf.core.export_lib import maybe_quantize_model\n"
    "result = {\n"
    "    'none': maybe_quantize_model('/tmp/model.tflite', 'none'),\n"
    "    'null': maybe_quantize_model('/tmp/model.tflite', None),\n"
    "}\n"
    "print('RESULT ' + json.dumps(result))\n"
)
check = subprocess.run([str(VENV_PY), "-c", probe], capture_output=True, text=True)
print(check.stdout[-1500:], check.stderr[-1000:])
payload = next((line for line in reversed(check.stdout.splitlines())
                if line.strip().startswith("RESULT ")), "")
result = json.loads(payload[len("RESULT "):]) if payload else {}
print("maybe_quantize_model:", result)
record("quantization probe ran", check.returncode == 0, f"rc={check.returncode}")
record("--quantization_recipe=none leaves the model untouched",
       result.get("none") == "/tmp/model.tflite", str(result.get("none")))
record("quantization_recipe=None leaves the model untouched",
       result.get("null") == "/tmp/model.tflite", str(result.get("null")))
MANIFEST["quantization"] = {"recipe": "none", "probe": result}
gate(start)


## 7b. Builder/API compatibility gate (before the expensive export)

`litert-torch==0.10.0` calls `litertlm_builder.TfLiteModelType.PREFILL_DECODE` while
packaging. That module-level enum was removed in `litert-lm-builder>=0.19`, so the
builder is pinned to **0.18.1**. This cell proves the symbol exists and that the
exporter's wrapper imports the same module object before any export work starts.


In [ ]:
start = len(CHECKS)
import subprocess
compat = subprocess.run(
    [str(VENV_PY), "-c", """
import inspect, json
import litert_lm_builder as pkg
from litert_lm_builder import litertlm_builder as sub
from litert_torch.generative.export_hf.core import litert_lm_builder as wrapper
print(json.dumps({
    'package_file': inspect.getsourcefile(pkg),
    'submodule_file': inspect.getsourcefile(sub),
    'package_has_enum': hasattr(pkg, 'TfLiteModelType'),
    'package_has_prefill': hasattr(getattr(pkg, 'TfLiteModelType', object), 'PREFILL_DECODE'),
    'submodule_has_enum': hasattr(sub, 'TfLiteModelType'),
    'wrapper_uses_package': wrapper.litertlm_builder is pkg,
    'wrapper_enum_is_same': getattr(wrapper.litertlm_builder, 'TfLiteModelType', object)
        is getattr(pkg, 'TfLiteModelType', object),
    'has_builder': hasattr(pkg, 'LitertLmFileBuilder'),
    'has_metadata': hasattr(pkg, 'Metadata'),
    'has_dtype': hasattr(pkg, 'DType'),
}))
"""],
    capture_output=True, text=True,
)
print(compat.stdout[-2000:], compat.stderr[-1000:])
record("compat probe ran", compat.returncode == 0, f"rc={compat.returncode}")

# transformers prints [ERROR]/warnings to stdout; the JSON is the last object line.
payload = next((line for line in reversed(compat.stdout.splitlines())
                if line.strip().startswith("{")), "")
info = json.loads(payload) if payload else {}
record("litert_lm_builder.TfLiteModelType exists", info.get("package_has_enum") is True)
record("TfLiteModelType.PREFILL_DECODE exists", info.get("package_has_prefill") is True)
record("builder submodule also exposes TfLiteModelType", info.get("submodule_has_enum") is True)
record("exporter wrapper imports the builder package", info.get("wrapper_uses_package") is True,
       info.get("package_file"))
record("wrapper sees the same TfLiteModelType", info.get("wrapper_enum_is_same") is True)
record("builder has LitertLmFileBuilder/Metadata/DType",
       all(info.get(k) is True for k in ("has_builder", "has_metadata", "has_dtype")))
MANIFEST["builder_compat"] = info
gate(start)


## 7. Official export (`export_hf`) inside the 3.11 venv

In [ ]:
import shutil, subprocess
from pathlib import Path

OUT = Path("/content/out-litertlm")
OUT.mkdir(exist_ok=True)
free_gb = shutil.disk_usage("/content").free / 1e9
record("free disk > 6 GB before conversion", free_gb > 6, f"{free_gb:.1f} GB")
assert free_gb > 6, "not enough disk for conversion"

EXPORT_CMD = [
    str(VENV_PY), "-m", "litert_torch.generative.export_hf",
    f"--model={CKPT_FIXED}",
    f"--litert_lm_llm_metadata_override={META}",
    "--quantization_recipe=none",
    "--model_dtype=float32",  # bf16 is rejected by the MLIR lowering (.numpy())
    f"--output_dir={OUT}",
]
print(" ".join(EXPORT_CMD))
result = subprocess.run(EXPORT_CMD, capture_output=True, text=True)
print("--- export stdout ---")
print(result.stdout[-6000:])
print("--- export stderr ---")
print(result.stderr[-6000:])
record("export_hf exit code 0", result.returncode == 0, f"rc={result.returncode}")
MANIFEST["export_command"] = EXPORT_CMD
if result.returncode != 0:
    raise SystemExit("export failed; see the captured log above")

## 8. Exactly one artifact, size and SHA-256

In [ ]:
import hashlib, json
from pathlib import Path

bundles = sorted(Path("/content/out-litertlm").rglob("*.litertlm"))
print("found:", [str(p) for p in bundles])
record("exactly one .litertlm artifact", len(bundles) == 1, f"{len(bundles)} found")
assert len(bundles) == 1, f"expected 1 bundle, found {len(bundles)}"
ARTIFACT = bundles[0]

size = ARTIFACT.stat().st_size
digest = hashlib.sha256(ARTIFACT.read_bytes()).hexdigest()
record("artifact under 1.5 GB (shipping limit)", size < 1_500_000_000, f"{size/1e9:.3f} GB")
MANIFEST["artifact_within_shipping_limit"] = size < 1_500_000_000
record("artifact non-empty", size > 1024, f"{size} bytes")
MANIFEST["artifact"] = {"path": str(ARTIFACT), "bytes": size,
                        "gb": round(size / 1e9, 3), "sha256": digest}
print(json.dumps(MANIFEST["artifact"], indent=2))
if size >= 1_500_000_000:
    print("NOTE: unquantized float32 exceeds the 1.5 GB shipping limit; this artifact is "
          "diagnostic only and must not be shipped.")

## 8b. Weight precision (no weight quantization)


In [ ]:
start = len(CHECKS)
import json, pathlib, subprocess, tempfile

# Inspect the exported graph: record tensor dtypes and prove there are no int8/uint8
# weights (i.e. no weight quantization).
UNPACK = ARTIFACT.parent / "unpacked-bundle"
subprocess.run([str(LM_VENV / "bin" / "litert-lm"), "unpack", str(ARTIFACT), "--output-dir", str(UNPACK), "--allow-overwrite"],
               check=True, capture_output=True)
tflite = next(UNPACK.rglob("*.tflite"))
probe = (
    "import json, sys\n"
    "from collections import Counter\n"
    "from ai_edge_litert.interpreter import Interpreter\n"
    "interp = Interpreter(model_path=sys.argv[1]); interp.allocate_tensors()\n"
    "types = Counter(str(t['dtype']) for t in interp.get_tensor_details())\n"
    "print(json.dumps({'file': sys.argv[1], 'tensor_dtypes': dict(types)}))\n"
)
inspect = subprocess.run([str(VENV_PY), "-c", probe, str(tflite)], capture_output=True, text=True)
print(inspect.stdout, inspect.stderr[-1000:])
record("graph tensor dtypes inspected", inspect.returncode == 0, f"rc={inspect.returncode}")
dtypes = json.loads(inspect.stdout).get("tensor_dtypes", {}) if inspect.returncode == 0 else {}
integer = sum(v for k, v in dtypes.items() if "int8" in k or "uint8" in k)
record("no int8/uint8 tensors (unquantized)", integer == 0, f"{integer} integer tensors")
record("float weights present", any("float" in k for k in dtypes), str(list(dtypes)))
MANIFEST["precision"] = {"tflite": str(tflite), "tensor_dtypes": dtypes,
                          "weight_quantization": "none", "model_dtype": "float32"}
gate(start)


## 9. Load with `litert-lm describe` (from the 3.11 venv)

In [ ]:
import subprocess

LITERT = str(LM_VENV / "bin" / "litert-lm")  # 0.19.1 runtime venv
describe = subprocess.run([LITERT, "describe", str(ARTIFACT)], capture_output=True, text=True)
print("--- describe stdout ---")
print(describe.stdout[:4000])
print("--- describe stderr ---")
print(describe.stderr[-2000:])
record("litert-lm describe exit code 0", describe.returncode == 0, f"rc={describe.returncode}")
record("litert-lm describe produced output", bool(describe.stdout.strip()), f"{len(describe.stdout)} chars")
MANIFEST["describe"] = {"returncode": describe.returncode, "stdout_chars": len(describe.stdout)}
if describe.returncode != 0:
    raise SystemExit("describe failed; the bundle is not loadable")

## 10. Token-level validation against the HF reference

In [ ]:
start = len(CHECKS)
import json, subprocess
from pathlib import Path

VALIDATOR = Path("/content/validate_bundle.py")
VALIDATOR.write_text(r'''"""Token-level acceptance harness for a .litertlm bundle (runtime venv only).

Requires: litert_lm (0.19.1) + a reference.json produced by tooling/hf_reference.py.

    python tooling/validate_bundle.py --bundle model.litertlm \
        --reference artifacts/eval-hf/reference.json --max-tokens 128
"""

from __future__ import annotations

import argparse
import json
import re
from collections import Counter
from pathlib import Path

import litert_lm as L

MAX_TOKENS = 128
NGRAW = 3


def words(text: str) -> set[str]:
    return set(re.findall(r"[a-z]{3,}", text.lower()))


def jaccard(a: str, b: str) -> float:
    left, right = words(a), words(b)
    return len(left & right) / len(left | right) if left and right else 0.0


def repeated_ngrams(text: str, n: int = NGRAW, limit: int = 3) -> int:
    tokens = re.findall(r"[a-z0-9]+", text.lower())
    grams = Counter(tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1))
    return max(grams.values()) if grams else 0


def main() -> int:
    parser = argparse.ArgumentParser()
    parser.add_argument("--bundle", required=True, type=Path)
    parser.add_argument("--reference", default=Path("artifacts/eval-hf/reference.json"), type=Path)
    parser.add_argument("--max-tokens", type=int, default=MAX_TOKENS)
    parser.add_argument("--out", default=Path("artifacts/eval-hf/bundle-validation.json"), type=Path)
    args = parser.parse_args()

    reference = json.loads(args.reference.read_text())["records"]
    engine = L.Engine(str(args.bundle), backend=L.Backend.CPU, max_num_tokens=2048)
    eos_ids = [item for group in engine.eos_token_ids for item in (group if isinstance(group, (list, tuple)) else [group])]

    results, failures = [], 0
    for record in reference:
        rendered = record["rendered_prompt"]
        prompt_ids = engine.tokenize(rendered)
        prompt_ids_match = prompt_ids == record["prompt_tokens"]

        session = engine.create_session(
            apply_prompt_template=False,
            sampler_config=L.SamplerConfig(temperature=0.0, top_k=1, top_p=1.0, seed=42),
            max_output_tokens=args.max_tokens,
        )
        session.run_prefill([rendered])
        response = session.run_decode()
        text = "".join(response.texts).strip()
        generated_ids = engine.tokenize(text)

        hf_ids = record["generated_ids"]
        # The HF reference includes the trailing EOS token; LiteRT ids come from
        # re-tokenizing text, so the EOS has no textual form. Strip it from both
        # before comparing, otherwise the only "divergence" is the final EOS slot.
        if hf_ids and hf_ids[-1] in eos_ids:
            hf_ids = hf_ids[:-1]
        if generated_ids and generated_ids[-1] in eos_ids:
            generated_ids = generated_ids[:-1]
        limit = min(20, len(hf_ids), len(generated_ids))
        first_divergence = next((i for i in range(limit) if hf_ids[i] != generated_ids[i]), None)
        if first_divergence is None and len(generated_ids) != len(hf_ids):
            first_divergence = limit

        lines = [line.strip() for line in text.splitlines() if line.strip()]
        repeated_line = max(Counter(lines).values()) if lines else 0
        ascii_ratio = sum(1 for ch in text if ord(ch) < 128) / max(len(text), 1)
        stopped_naturally = len(generated_ids) < args.max_tokens - 1

        checks = {
            "prompt_ids_identical": prompt_ids_match,
            "first_20_ids_match": first_divergence is None,
            "eos_generated_naturally": stopped_naturally,
            "english": ascii_ratio >= 0.8,
            "no_sentence_loop": repeated_line <= 3,
            "no_ngram_loop": repeated_ngrams(text) <= 3,
            "semantic_match": jaccard(text, record["text"]) >= 0.35,
        }
        failed = [name for name, ok in checks.items() if not ok]
        failures += len(failed)
        results.append({
            "prompt": record["prompt"],
            "checks": checks,
            "failed": failed,
            "first_divergence_index": first_divergence,
            "hf_first_20_ids": hf_ids[:20],
            "litert_first_20_ids": generated_ids[:20],
            "hf_first_20_tokens": [engine.detokenize([i]) for i in hf_ids[:20]],
            "litert_first_20_tokens": [engine.detokenize([i]) for i in generated_ids[:20]],
            "termination": {
                "generated_tokens": len(generated_ids),
                "max_tokens": args.max_tokens,
                "eos_generated": stopped_naturally,
                "eos_ids": eos_ids,
            },
            "hf_text": record["text"],
            "litert_text": text,
            "ascii_ratio": round(ascii_ratio, 3),
            "max_repeated_line": repeated_line,
            "max_repeated_3gram": repeated_ngrams(text),
            "jaccard_vs_hf": round(jaccard(text, record["text"]), 3),
        })
        print(f"\n=== {record['prompt'][:46]}")
        print(f"  first divergence: {first_divergence} | tokens {len(generated_ids)}/{args.max_tokens}")
        print(f"  failed: {failed or 'none'}")
        print(f"  litert: {text[:160]!r}")

    summary = {"bundle": str(args.bundle), "max_tokens": args.max_tokens,
               "results": results, "failures": failures}
    args.out.parent.mkdir(parents=True, exist_ok=True)
    args.out.write_text(json.dumps(summary, indent=2) + "\n")
    print(f"\nwrote {args.out}; failed checks: {failures}")
    return 0 if failures == 0 else 1


if __name__ == "__main__":
    raise SystemExit(main())
''')

# 6b writes hf_reference.json; older runs put "records" inside audit.json.
REFERENCE = Path("/content/hf_reference.json")
if not REFERENCE.exists():
    alt = Path("/content/audit.json")
    if alt.exists():
        data = json.loads(alt.read_text())
        if isinstance(data, dict) and "records" in data:
            REFERENCE.write_text(json.dumps({"records": data["records"]}, indent=2))
record("HF reference file present", REFERENCE.exists(), str(REFERENCE))
if not REFERENCE.exists():
    raise SystemExit("no HF reference: re-run 6b (checkpoint audit) first")

validation = subprocess.run(
    [str(LM_PY), str(VALIDATOR), "--bundle", str(ARTIFACT),
     "--reference", str(REFERENCE),
     "--max-tokens", "128", "--out", "/content/bundle_validation.json"],
    capture_output=True, text=True,
)
print(validation.stdout[-6000:])
print(validation.stderr[-3000:])
record("token-level validation ran", validation.returncode == 0, f"rc={validation.returncode}")
report_path = Path("/content/bundle_validation.json")
report = json.loads(report_path.read_text()) if report_path.exists() else {}
for item in report.get("results", []):
    tag = item["prompt"][:34]
    for name, ok in item["checks"].items():
        record(f"{tag}: {name}", ok, f"divergence={item['first_divergence_index']}")
    record(f"{tag}: EOS within 128 tokens", item["termination"]["eos_generated"],
           f"{item['termination']['generated_tokens']} tokens")
    print(f"{tag}: litert={item['litert_text'][:120]!r} hf={item['hf_text'][:120]!r}")
record("three HF references compared", len(report.get("results", [])) == 3,
       f"{len(report.get('results', []))}")
MANIFEST["bundle_validation"] = report
gate(start)


## 11. Persist the artifact and manifest

Colab sessions are ephemeral. When Drive is mounted the artifact and `manifest.json` are
copied there; otherwise they are written to `/content` for download.


In [ ]:
import json, shutil
from pathlib import Path

MANIFEST["generated_at"] = datetime.datetime.now(datetime.timezone.utc).isoformat()
MANIFEST["notebook"] = "colab/MediRead_Qwen3_LiteRT_Export_colab.ipynb"

if "drive_ok" in globals() and drive_ok:
    target = Path("/content/drive/MyDrive/mediread/litertlm")
else:
    target = Path("/content/mediread/litertlm")
target.mkdir(parents=True, exist_ok=True)
shutil.copy2(ARTIFACT, target / ARTIFACT.name)
manifest_path = target / "manifest.json"
manifest_path.write_text(json.dumps(MANIFEST, indent=2))
record("artifact and manifest saved", True, str(target))
print("artifact:", target / ARTIFACT.name)
print("manifest:", manifest_path)
print(json.dumps(MANIFEST, indent=2)[:3000])

## 12. Acceptance summary

Every check must be PASS. A FAIL means the artifact must not be used.

In [ ]:
# The 1.5 GB limit is a shipping constraint, not a conversion-correctness check.
# For this unquantized experiment an over-size artifact is expected and diagnostic.
INFORMATIONAL = {"artifact under 1.5 GB (shipping limit)"}

print(f"{'status':6} check")
print("-" * 72)
for item in CHECKS:
    print(f"{item['status']:6} {item['name']}" + (f"  [{item['detail']}]" if item["detail"] else ""))

failed = [c for c in CHECKS if c["status"] == "FAIL" and c["name"] not in INFORMATIONAL]
warnings = [c for c in CHECKS if c["status"] == "FAIL" and c["name"] in INFORMATIONAL]
MANIFEST["checks"] = CHECKS
MANIFEST["warnings"] = [c["name"] for c in warnings]
MANIFEST["shippable"] = not warnings
MANIFEST["notebook_status"] = ("PASSED_ALL_CHECKS" if not failed and not warnings
                               else "PASSED_DIAGNOSTIC_ONLY" if not failed
                               else "FAILED_CHECKS")
print("\n" + "=" * 72)
print(f"notebook_status: {MANIFEST['notebook_status']}  "
      f"({len(CHECKS) - len(failed) - len(warnings)}/{len(CHECKS)} passed, "
      f"{len(warnings)} informational)")
if warnings:
    print("INFORMATIONAL:", ", ".join(c["name"] for c in warnings),
          "— artifact is diagnostic only, do not ship")
if failed:
    print("FAILED:", ", ".join(c["name"] for c in failed))
    raise SystemExit("one or more acceptance checks failed; do not use this artifact")

print("Artifact:", MANIFEST["artifact"]["path"])
print("SHA-256 :", MANIFEST["artifact"]["sha256"])
print("Size    :", f"{MANIFEST['artifact']['bytes']} bytes ({MANIFEST['artifact']['gb']:.3f} GB)")
print("Shippable:", MANIFEST["shippable"])

print("Artifact:", MANIFEST["artifact"]["path"])
print("SHA-256 :", MANIFEST["artifact"]["sha256"])
print("Size    :", f"{MANIFEST['artifact']['bytes']} bytes ({MANIFEST['artifact']['gb']:.3f} GB)")
print("Shippable:", MANIFEST["shippable"])

# Re-save the manifest so it contains checks/status (11 ran before this cell).
from pathlib import Path
target = (Path("/content/drive/MyDrive/mediread/litertlm") if globals().get("drive_ok")
          else Path("/content/mediread/litertlm"))
target.mkdir(parents=True, exist_ok=True)
(target / "manifest.json").write_text(json.dumps(MANIFEST, indent=2))
print("manifest updated:", target / "manifest.json")
